In [1]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

import matplotlib.pyplot as plt
import numpy as np
import anndata as ad
import mudata
import celldega as dega
import scanpy as sc
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon
import warnings
warnings.filterwarnings("ignore")


env: ANYWIDGET_HMR=1


In [2]:
def calc_gene_expression_by_nbhd(
    adata,
    gdf_nbhd: gpd.GeoDataFrame,
    unique_nbhd_col: str = "name"
) -> gpd.GeoDataFrame:
    """
    Calculate mean gene expression per band from adata and band GeoDataFrame.
    Args:
        adata: AnnData object with spatial info in .obsm['spatial'] and gene data in .X.
        gdf_bands: GeoDataFrame with band polygons, must contain a 'band' column.
        unique_nbhd_col: Neighborhood column in gdf_bands that labels each nbhd.
    Returns:
        GeoDataFrame with band geometries and mean gene expression values
        as columns named like 'GENE_mean'.
    """


    gdf_nbhd_join = gdf_nbhd.copy()
    # Get gene list
    gene_list = adata.var.index

    gene_exp = pd.DataFrame(
        data=adata[:, gene_list].X.toarray() if hasattr(adata.X, 'toarray') else adata[:, gene_list].X,
        columns=gene_list,
        index=adata.obs_names
    )

    # Create combined DataFrame with geometry
    gdf_cell = gpd.GeoDataFrame(
        data={
            'cluster': adata.obs['leiden'],
            **gene_exp  # Unpacks all gene columns
        },
        geometry=[Point(xy) for xy in adata.obsm['spatial'][:, :2]],
        crs="EPSG:4326"  # Set your coordinate system here
    )

    # Spatial join: Assign each cell to the closest buffer
    gdf_join_all = gdf_cell.sjoin(gdf_nbhd, how="left", predicate="within").drop(
        columns=['index_right'], errors='ignore')

    # Compute mean expression for each gene
    for gene in gene_list:
        band_avg_expression = gdf_join_all.groupby(unique_nbhd_col)[gene].mean().reset_index()
        band_avg_expression.columns = [unique_nbhd_col, f"{gene}"]
        gdf_nbhd_join = gdf_nbhd_join.merge(band_avg_expression, on=unique_nbhd_col)

    # Rename unique_nbhd_col to 'nbhd_id'
    gdf_nbhd_join.rename(columns={unique_nbhd_col: 'nbhd_id'}, inplace=True)

    df_nbhd_join = pd.DataFrame(gdf_nbhd_join.drop(columns="geometry"))

    print (gdf_nbhd_join.columns)


    # Convert to anndata
    gdf_nbhd_join.set_index('nbhd_id', inplace=True)
    obs_cols = ['cat', 'inv_alpha', 'area', 'band_width']
    obs_cols = [col for col in obs_cols if col in gdf_nbhd_join.columns]
    obs = gdf_nbhd_join[obs_cols]
    X = gdf_nbhd_join[gene_list].values
    var = pd.DataFrame(index=gene_list)
    adata = ad.AnnData(X=X, obs=obs, var=var)      

    # return gdf_nbhd_join, df_nbhd_join, adata
    return adata



def generate_hex_grid(gdf_cell, radius=20):
    # 1. Get the convex hull of all points
    bounding_geom = gdf_cell.unary_union.convex_hull

    # 2. Determine bounding box
    minx, miny, maxx, maxy = bounding_geom.bounds

    # 3. Calculate vertical spacing
    dx = np.sqrt(3) * radius  # horizontal distance between centers
    dy = 1.5 * radius         # vertical distance between centers

    # 4. Create hexagons
    hexagons = []
    row = 0
    y = miny
    while y < maxy + dy:
        x_offset = 0 if row % 2 == 0 else dx / 2
        x = minx
        while x < maxx + dx:
            cx = x + x_offset
            cy = y
            hexagon = create_hexagon(cx, cy, radius)
            if hexagon.intersects(bounding_geom):
                hexagons.append(hexagon)
            x += dx
        y += dy
        row += 1

    return gpd.GeoDataFrame({
        'name': [f'hex_{i}' for i in range(len(hexagons))],
        'geometry': hexagons
    }, crs=gdf_cell.crs)

def create_hexagon(x_center, y_center, radius):
    """Create a vertically-oriented (pointy-topped) hexagon."""
    angles_deg = [30 + i * 60 for i in range(6)]
    angles_rad = [np.radians(a) for a in angles_deg]
    points = [
        (x_center + radius * np.cos(a), y_center + radius * np.sin(a))
        for a in angles_rad
    ]
    return Polygon(points)



def calc_grad_nbhd_from_roi(polygon, gdf_reference, band_width):
    """
    Create concentric rings (buffers) around a given polygon.
    This function generates multiple concentric rings around a polygon by creating buffers
    at increasing distances from the original polygon. Each ring is defined by the area
    between two consecutive buffers.
    Parameters:
    polygon (shapely.geometry.Polygon): The polygon around which to create concentric rings.
    gdf_reference (geopandas.GeoDataFrame): A reference GeoDataFrame indicating the boudnary.
    band_width (float): The width of each ring in the same units as the polygon.
    Returns:
    geopandas.GeoDataFrame: A GeoDataFrame containing the concentric rings as individual
                            geometries with a 'band' identifier for each ring. The GeoDataFrame
                            is set to the EPSG:4326 coordinate reference system.
    """
    band_list = []
    for i in range(1, 30 + 1):
        outer = polygon.buffer(i * band_width)
        inner = polygon.buffer((i - 1) * band_width)
        ring = outer.difference(inner)

        band = polygon.copy()
        band["geometry"] = ring
        band["band"] = i
        band_list.append(band)

    # Compare with the reference GeoDataFrame and only keep the ones that intersect
    band_list = [
        band[band.geometry.intersects(gdf_reference.unary_union)]
        for band in band_list
    ]

    return gpd.GeoDataFrame(pd.concat(band_list, ignore_index=True)).set_crs('EPSG:4326')


def calc_grad_nbhd_from_roi_new(polygon, gdf_reference, band_width=300):
    """
    Generate concentric rings (neighborhood bands) from a polygon,
    clipped to the convex hull of a reference GeoDataFrame.
    
    Parameters:
    -----------
    polygon : GeoDataFrame
        GeoDataFrame containing a single polygon
    gdf_reference : GeoDataFrame
        Reference GeoDataFrame used to calculate the boundary area (convex hull)
    band_width : float
        Width of each band in microns (default: 300)
    
    Returns:
    --------
    GeoDataFrame
        GeoDataFrame with columns for band (index of ring) and geometry (polygon)
    """
    if len(polygon) != 1:
        raise ValueError("Input polygon GeoDataFrame must contain exactly one polygon")
    
    roi_polygon = polygon.geometry.iloc[0]
    boundary = gdf_reference.unary_union.convex_hull

    bands = []
    current_polygon = roi_polygon
    band_idx = 0

    # Add the original polygon as band 0
    bands.append({'band': band_idx, 'geometry': roi_polygon})

    while True:
        band_idx += 1
        # Generate next ring
        next_buffer = current_polygon.buffer(band_width)
        ring = next_buffer.difference(current_polygon)

        # Clip the ring to the convex hull boundary
        ring_clipped = ring.intersection(boundary)

        # Stop if no part of the ring remains within boundary
        if ring_clipped.is_empty:
            break

        bands.append({'band': band_idx, 'geometry': ring_clipped})
        current_polygon = next_buffer

    gdf = gpd.GeoDataFrame(bands, crs=polygon.crs)
    gdf['band_width'] = band_width

    return gdf

In [3]:
# dataset_name = 'Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs'
dataset_name = 'Xenium_V1_human_Pancreas_FFPE_outs'
tech = 'Xenium'
DATA_DIR = f'data/raw'
data_dir = f'{DATA_DIR}/{dataset_name}'
path_landscape_files = f'data/landscape_files/{dataset_name}'

# Load h5ad file
adata = sc.read_h5ad(f'{data_dir}.h5ad')
adata.obs.set_index('cell_id', inplace=True)

# Derive the cell metadata
df_cell = gpd.GeoDataFrame(
    data={
        'cluster': adata.obs['leiden'],
        'x': adata.obsm['spatial'][:, 0],
        'y': adata.obsm['spatial'][:, 1],
    },
)

# Calculate alpha shapes in micron space
df_cell['geometry'] = df_cell.apply(lambda row: [round(row['x'],3), round(row['y'], 3)], axis=1)


# Create combined DataFrame with geometry
gdf_cell = gpd.GeoDataFrame(
    data={
        'cluster': adata.obs['leiden'],
    },
    geometry=[Point(xy) for xy in adata.obsm['spatial'][:, :2]],
    crs="EPSG:4326"
)

## NBHD: ALPA

In [4]:
alphas_list = range(30, 60, 5)
gdf_alpha = dega.nbhd.alpha_shape_cell_clusters(df_cell, cat='cluster', alphas=alphas_list)
gdf_alpha_select = gdf_alpha.loc[gdf_alpha['inv_alpha']==35]
gdf_alpha_join = gdf_alpha_select.copy()
adata_alph = calc_gene_expression_by_nbhd(adata, gdf_alpha_select, 'name')



# fig, ax = plt.subplots(1, 2, figsize=(20, 10))
# gdf_join_all.plot('cat', ax=ax[0], aspect=1, cmap='tab20', edgecolor='white', alpha=0.8)
# ax[0].set_title('Alpha shapes colored by cell clusters')
# ax[0].invert_yaxis()                            
# gdf_join_all.plot('EPCAM', ax=ax[1], aspect=1, cmap='coolwarm', edgecolor='white', alpha=0.8)
# ax[1].set_title('EPCAM expression aggregated by nbhd (mean), nbhd type: ALPH')
# ax[1].invert_yaxis()
# plt.show()

Index(['nbhd_id', 'cat', 'geometry', 'inv_alpha', 'area', 'ABCC11', 'ACE2',
       'ACKR1', 'ACTA2', 'ACTG2',
       ...
       'TRAC', 'TREM2', 'TSPAN19', 'UBE2C', 'UMOD', 'UPK3B', 'VCAN', 'VSIG4',
       'VWA5A', 'VWF'],
      dtype='object', length=382)


## NBHD: HEX


In [5]:
# Example usage:
gdf_hex = generate_hex_grid(gdf_cell, radius=50)
adata_hex = calc_gene_expression_by_nbhd(adata, gdf_hex, 'name')

# fig, ax = plt.subplots(1, 2, figsize=(20, 10))
# gdf_hex.plot(ax=ax[0], aspect=1, edgecolor='blue', facecolor='none', linewidth=0.5)
# ax[0].set_title('Hex tiles covering the sample')
# ax[0].invert_yaxis()                            
# gdf_join_all.plot('EPCAM_mean', ax=ax[1], aspect=1, cmap='coolwarm', edgecolor='white', alpha=0.8)
# ax[1].set_title('EPCAM expression aggregated by nbhd (mean), nbhd type: HEX')
# ax[1].invert_yaxis()
# plt.show()

Index(['nbhd_id', 'geometry', 'ABCC11', 'ACE2', 'ACKR1', 'ACTA2', 'ACTG2',
       'ADAM28', 'ADAMTS1', 'ADGRE1',
       ...
       'TRAC', 'TREM2', 'TSPAN19', 'UBE2C', 'UMOD', 'UPK3B', 'VCAN', 'VSIG4',
       'VWA5A', 'VWF'],
      dtype='object', length=379)


## NBHD: GRAD

In [6]:
gdf_micron = gpd.read_parquet(f'{path_landscape_files}/spatial_regions/sketched_regions_micron.parquet')
gdf_roi = gdf_micron.loc[gdf_micron['roi'] == 'region_1']
gdf_bands = calc_grad_nbhd_from_roi_new(gdf_roi, gdf_cell, 200)
adata_grad = calc_gene_expression_by_nbhd(adata, gdf_bands, 'band')

# fig, ax = plt.subplots(1, 2, figsize=(20, 10))
# gdf_cell.plot('cluster', ax=ax[0], aspect=1, edgecolor='none',linewidth=0.1, alpha=0.3, markersize=5)
# gdf_bands.plot('band', ax=ax[0], aspect=1, cmap='terrain', edgecolor='white', alpha=0.8)
# ax[0].set_title('Concentric rings colored by cell clusters')
# ax[0].invert_yaxis()
# gdf_cell.plot('cluster', ax=ax[1], aspect=1, edgecolor='none',linewidth=0.1, alpha=0.3, markersize=5)                            
# gdf_join_all.plot('EPCAM_mean', ax=ax[1], aspect=1, cmap='coolwarm', edgecolor='white', alpha=0.8)
# ax[1].set_title('EPCAM expression aggregated by nbhd (mean), nbhd type: GRAD')
# ax[1].invert_yaxis()
# plt.show()

Index(['nbhd_id', 'geometry', 'band_width', 'ABCC11', 'ACE2', 'ACKR1', 'ACTA2',
       'ACTG2', 'ADAM28', 'ADAMTS1',
       ...
       'TRAC', 'TREM2', 'TSPAN19', 'UBE2C', 'UMOD', 'UPK3B', 'VCAN', 'VSIG4',
       'VWA5A', 'VWF'],
      dtype='object', length=380)


In [7]:
mdata = mudata.MuData({'ALPH': adata_alph, 'HEX': adata_hex, 'GRAD': adata_grad})
mdata.write(f"{path_landscape_files}/nhbd_collections.h5mu")  

In [8]:
import muon as mu

# Load the h5mu file
mdata = mu.read(f"{path_landscape_files}/nhbd_collections.h5mu")

In [9]:
# import scanpy as sc

# nbhd_type = 'HEX'

# sc.pp.normalize_total(mdata.mod[nbhd_type])
# sc.pp.log1p(mdata.mod[nbhd_type])
# sc.pp.pca(mdata.mod[nbhd_type])
# sc.pp.neighbors(mdata.mod[nbhd_type])


### ChatGPT notes

In [ ]:
class NBHD:

    def __init__(self, gdf, nbhd_type, source=None, name=None, meta=None):

        """

        Parameters

        ----------

        gdf : geopandas.GeoDataFrame

            A GeoDataFrame with one row per neighborhood. Must have a 'geometry' column.

        nbhd_type : str

            One of: 'SKTCH', 'HEX', 'ALPH', 'GRAD'

        source : str or dict, optional

            Optional description of where this neighborhood set came from (e.g., 'B cells', clustering params)

        name : str, optional

            A name or label for this neighborhood set.

        meta : dict, optional

            Any other user-defined metadata to store.

        """

        self.gdf = gdf.copy()

        self.nbhd_type = nbhd_type

        self.source = source

        self.name = name

        self.meta = meta or {}



        # Store all derived high-dimensional data here

        self.derived = {

            'NBI': None,

            'NBG-CF': None,

            'NBG-CD': None,

            'NBG-LCD': {},  # keyed by cluster name

            'NBP': None,

            'NBN-O': None,

            'NBN-B': None,

        }



    def set_derived(self, key, data, subkey=None):

        """

        Set a derived data matrix.



        Parameters

        ----------

        key : str

            One of 'NBI', 'NBG-CF', 'NBG-CD', 'NBG-LCD', 'NBP', 'NBN-O', 'NBN-B'

        data : pd.DataFrame or np.ndarray

            Matrix with shape [n_neighborhoods x n_features]

        subkey : str, optional

            For NBG-LCD or other nested structures, store under a subkey (e.g., cluster name).

        """

        if key == 'NBG-LCD':

            assert subkey is not None, "NBG-LCD requires a subkey (e.g., cluster name)"

            self.derived[key][subkey] = data

        else:

            self.derived[key] = data



    def get_derived(self, key, subkey=None):

        if key == 'NBG-LCD':

            return self.derived[key].get(subkey)

        return self.derived.get(key)



    def to_geodataframe(self):

        """Return the underlying GeoDataFrame."""

        return self.gdf



    def summary(self):

        return {

            "name": self.name,

            "type": self.nbhd_type,

            "n_regions": len(self.gdf),

            "derived": {k: self._derived_summary(k) for k in self.derived},

            "meta": self.meta,

        }



    def _derived_summary(self, key):

        val = self.derived[key]

        if val is None:

            return None

        if isinstance(val, dict):

            return {k: v.shape for k, v in val.items()}

        return val.shape



# # Create a neighborhood set from alpha shapes
# nbhd = NBHD(alpha_gdf, nbhd_type='ALPH', source='B cells', name='Bcell alpha shapes')

# # Add derived gene expression
# nbhd.set_derived('NBG-CD', gene_expr_df)

# # Add cluster-specific gene expression
# nbhd.set_derived('NBG-LCD', gene_expr_df_cd8, subkey='CD8_T')

# # Retrieve and visualize
# df = nbhd.get_derived('NBG-LCD', 'CD8_T')


